<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_13_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 📘 **Module 13 — Exploratory Data Analysis** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 13 — Exploratory Data Analysis (EDA)
*IBM Data Analysis · Module 3 of 6 (Module 13 of 16)*

EDA is the **chart-driven hunt for patterns** before any modelling. The goal is to walk away with three sentences:

> "Here's how the target is distributed. Here are the 2-3 features that matter most. Here's the weird thing we should fix or watch."

### What you'll cover
1. Univariate analysis — distribution of each variable
2. Bivariate — relationships between pairs
3. Correlation matrices & heatmaps
4. Group-by analysis (categorical → numeric)
5. Pivot tables — two-dimensional group-bys
6. Pair plots & scatter matrices
7. Practice


In [ ]:
!pip -q install pandas numpy seaborn matplotlib scipy
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns

url  = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
cols = ["mpg","cylinders","displacement","horsepower","weight",
        "acceleration","model_year","origin","car_name"]
cars = pd.read_csv(url, sep=r"\s+", names=cols, na_values="?")
cars["horsepower"] = cars["horsepower"].fillna(cars["horsepower"].median())
cars["origin"] = cars["origin"].map({1:"USA",2:"Europe",3:"Japan"}).astype("category")
cars.head()

## 1. Univariate — one variable at a time

For each numeric column, look at its distribution. For each categorical column, look at the value counts.

In [ ]:
# Numeric distributions in one figure
fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for ax, col in zip(axes.flat, ["mpg","horsepower","weight","displacement","acceleration","model_year"]):
    cars[col].plot.hist(bins=20, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(col)
plt.tight_layout(); plt.show()

print(cars.describe().round(2).T)

In [ ]:
# Categorical: origin
sns.countplot(data=cars, x="origin"); plt.title("Cars per origin"); plt.show()
print(cars["origin"].value_counts())

## 2. Bivariate — pairs of variables

Three useful pair patterns:

| X type | Y type | Chart |
|---|---|---|
| numeric | numeric | scatter (+ regplot for trend) |
| categorical | numeric | boxplot or violin |
| categorical | categorical | crosstab + heatmap |

In [ ]:
# numeric vs numeric
sns.regplot(data=cars, x="weight", y="mpg", scatter_kws={"alpha":.4},
            line_kws={"color":"red"})
plt.title("Heavier cars get worse mpg — strong negative slope"); plt.show()

In [ ]:
# categorical vs numeric
sns.boxplot(data=cars, x="origin", y="mpg")
plt.title("Japanese cars have the highest mpg distribution"); plt.show()

# violin = box + density
sns.violinplot(data=cars, x="origin", y="mpg"); plt.title("Same data, violin"); plt.show()

In [ ]:
# categorical vs categorical
ct = pd.crosstab(cars["origin"], cars["cylinders"])
print(ct)
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues")
plt.title("Counts: origin vs cylinders"); plt.show()

## 3. Correlation — which features move together?

Correlation $r \in [-1, 1]$:
- $r = +1$ → perfect positive
- $r = -1$ → perfect negative
- $r \approx 0$ → no linear relationship

⚠️ Correlation only catches **linear** relationships. Always look at scatter plots too.

In [ ]:
corr = cars.select_dtypes("number").corr()
print(corr.round(2))

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, ax=ax)
ax.set_title("Correlation heatmap"); plt.show()

### Pearson vs Spearman
- **Pearson** (default) — linear correlation.
- **Spearman** — rank correlation, robust to non-linear monotonic relationships and outliers.

In [ ]:
print("Pearson:")
print(cars[["mpg","weight","horsepower"]].corr(method="pearson").round(2))
print("\nSpearman (rank):")
print(cars[["mpg","weight","horsepower"]].corr(method="spearman").round(2))

## 4. Group-by analysis

In [ ]:
# Average mpg per origin
g = cars.groupby("origin", observed=True)["mpg"].agg(["mean","median","std","count"])
print(g.round(2))

# Multiple aggs at once
print(cars.groupby("cylinders").agg(
    avg_mpg=("mpg","mean"),
    avg_weight=("weight","mean"),
    n=("mpg","count"),
).round(1))

## 5. Pivot tables — group-by on two dimensions

In [ ]:
pivot = cars.pivot_table(values="mpg", index="origin", columns="cylinders",
                          aggfunc="mean", observed=True)
print(pivot.round(1))

sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Average mpg by origin × cylinders"); plt.show()

## 6. Pair plot / scatter matrix — see all pairs at once

A grid of scatterplots, one per pair of numeric columns; histograms on the diagonal. Best for small numbers of variables (<8).

In [ ]:
sns.pairplot(cars[["mpg","weight","horsepower","acceleration","origin"]],
             hue="origin", corner=True, plot_kws={"alpha":.5})
plt.show()

## 7. Putting it together — the EDA report

After running the above, three takeaways for the auto-mpg dataset:

1. **Distribution of mpg** is roughly bell-shaped, mean ≈ 23, range 9–47.
2. **Strongest predictors** of mpg are `weight` (r ≈ -0.83) and `horsepower` (r ≈ -0.78). Both negative — heavier and more powerful → worse mpg.
3. **Origin matters**: Japanese cars cluster around 30 mpg, USA around 20. Worth keeping `origin` as a feature.

That's the EDA done. Modelling next.

## 8. Practice

1. Plot the distribution of `mpg` split by `origin` (overlay or facet).
2. Compute the correlation between `mpg` and each numeric feature, sorted from strongest to weakest negative.
3. Build a pivot table of `mean horsepower` by `origin` × decade (use `(model_year//10)*10`).
4. Use `pairplot` on just `["mpg","weight","horsepower","acceleration"]`. Which pair has the cleanest relationship?


In [ ]:
# 1)
sns.kdeplot(data=cars, x="mpg", hue="origin", fill=True, alpha=.4)
plt.title("mpg by origin"); plt.show()

# 2)
print(cars.select_dtypes("number").corrwith(cars["mpg"]).sort_values())

# 3)
cars["decade"] = (cars["model_year"] // 10) * 10
print(cars.pivot_table(values="horsepower", index="origin", columns="decade",
                       aggfunc="mean", observed=True).round(1))

# 4)
sns.pairplot(cars[["mpg","weight","horsepower","acceleration"]], corner=True,
             plot_kws={"alpha":.5}); plt.show()

## Recap

✅ Univariate look at every column
✅ Bivariate by data-type pairs (num/num, cat/num, cat/cat)
✅ Pearson + Spearman correlations and the heatmap view
✅ Group-by + pivot table for two-dimensional summaries
✅ Pair plot for the all-pairs view

### Next
**Module 14 — Model Development** (regression — your first ML model).